In [1]:
import pandas as pd
#leggo il file e vedo le righe
df_prezzi=pd.read_csv(r"C:\Users\rauso\Desktop\tesi\dataset\dogecoin_prices_2021.csv")
print(df_prezzi.head())
print(""
"---------" \
"le info sono:" \
"-------------" \
"")
print(df_prezzi.info())


          open_time     price
0  01/01/2021 00:00  0.004672
1  01/01/2021 00:01  0.004673
2  01/01/2021 00:02  0.004686
3  01/01/2021 00:03  0.004671
4  01/01/2021 00:04  0.004676
---------le info sono:-------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 523168 entries, 0 to 523167
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   open_time  523168 non-null  object 
 1   price      523168 non-null  float64
dtypes: float64(1), object(1)
memory usage: 8.0+ MB
None


In [2]:
# --- Cella 2: Conversione Timestamp (UTC) ---

colonna_timestamp = 'open_time'
colonna_prezzo = 'price' # La colonna 'price' è già un float, perfetta.

# 1. Stampa il tipo di dato attuale per verifica
print(f"Tipo di dato ORIGINALE di '{colonna_timestamp}': {df_prezzi[colonna_timestamp].dtype}")

# 2. Converte la colonna in oggetti datetime (pandas)
#    Specifichiamo il 'format' esatto GG/MM/AAAA HH:MM
df_prezzi[colonna_timestamp] = pd.to_datetime(
    df_prezzi[colonna_timestamp], 
    format='%d/%m/%Y %H:%M'
)



Tipo di dato ORIGINALE di 'open_time': object


In [3]:
# 3. Assegna il fuso orario UTC (localize)
print("Timestamp convertiti, assegno fuso UTC...")
df_prezzi[colonna_timestamp] = df_prezzi[colonna_timestamp].dt.tz_localize('UTC')

# --- Cella 3: Impostazione Indice ---
# Impostiamo la colonna 'open_time' come indice del DataFrame
# Questo è FONDAMENTALE per l'analisi di serie storiche
df_prezzi = df_prezzi.set_index(colonna_timestamp)

# 4. Controlla il risultato
print("\nConversione e impostazione Indice completata.")
print(df_prezzi.head())

print("\nNuove informazioni sul DataFrame:")
df_prezzi.info()

Timestamp convertiti, assegno fuso UTC...

Conversione e impostazione Indice completata.
                              price
open_time                          
2021-01-01 00:00:00+00:00  0.004672
2021-01-01 00:01:00+00:00  0.004673
2021-01-01 00:02:00+00:00  0.004686
2021-01-01 00:03:00+00:00  0.004671
2021-01-01 00:04:00+00:00  0.004676

Nuove informazioni sul DataFrame:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 523168 entries, 2021-01-01 00:00:00+00:00 to 2021-12-31 00:00:00+00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   price   523168 non-null  float64
dtypes: float64(1)
memory usage: 8.0 MB


In [4]:
# --- Cella 4: Controllo e Riempimento Minuti Mancanti (Gap) ---

# 1. Definiamo l'indice temporale COMPLETO che ci aspettiamo
#    Dal primo timestamp all'ultimo, con frequenza 'min' (minuto)
indice_completo = pd.date_range(
    start=df_prezzi.index.min(), 
    end=df_prezzi.index.max(), 
    freq='min',
    tz='UTC' # Assicurati che anche il nuovo indice sia UTC
)

print(f"Minuti attesi nell'intervallo: {len(indice_completo)}")
print(f"Minuti presenti nei dati: {len(df_prezzi)}")
print(f"Minuti mancanti: {len(indice_completo) - len(df_prezzi)}")

# 2. Riaallinea i dati all'indice completo
#    Questo crea NaN (buchi) dove i minuti mancano
df_completo = df_prezzi.reindex(indice_completo)

# 3. Riempi i buchi (NaN) con l'ultimo valore valido
#    'ffill' = "forward fill"
df_riempito = df_completo.fillna(method='ffill')

# 4. (Opzionale) Controllo se sono rimasti NaN all'inizio (se il primo minuto mancava)
#    'bfill' = "backward fill", riempie all'indietro
df_riempito = df_riempito.fillna(method='bfill')

print("\nDati riempiti (head e tail per controllo):")
print(df_riempito.head())
print(df_riempito.tail())

print("\nControllo finale NaN:")
print(f"NaN rimasti: {df_riempito['price'].isna().sum()}")

# 5. Sovrascriviamo il vecchio DataFrame con quello pulito e completo
df_prezzi = df_riempito

Minuti attesi nell'intervallo: 524161
Minuti presenti nei dati: 523168
Minuti mancanti: 993

Dati riempiti (head e tail per controllo):
                              price
2021-01-01 00:00:00+00:00  0.004672
2021-01-01 00:01:00+00:00  0.004673
2021-01-01 00:02:00+00:00  0.004686
2021-01-01 00:03:00+00:00  0.004671
2021-01-01 00:04:00+00:00  0.004676
                            price
2021-12-30 23:56:00+00:00  0.1711
2021-12-30 23:57:00+00:00  0.1712
2021-12-30 23:58:00+00:00  0.1711
2021-12-30 23:59:00+00:00  0.1712
2021-12-31 00:00:00+00:00  0.1710

Controllo finale NaN:
NaN rimasti: 0


C:\Users\rauso\AppData\Local\Temp\ipykernel_19664\85212330.py:22: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_riempito = df_completo.fillna(method='ffill')
C:\Users\rauso\AppData\Local\Temp\ipykernel_19664\85212330.py:26: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_riempito = df_riempito.fillna(method='bfill')


In [5]:
# --- Cella 4.5: Filtro per tenere solo Marzo e Aprile ---

# Definiamo le date di inizio e fine
# (l'indice Datetime di pandas gestisce bene le date finali)
start_date = '2021-03-01'
end_date = '2021-04-30'

# Filtriamo il DataFrame usando .loc[] sull'indice
# Usiamo .copy() per creare un nuovo DataFrame indipendente
df_mar_apr = df_prezzi.loc[start_date:end_date].copy()

print(f"Righe totali nel DataFrame completo: {len(df_prezzi)}")
print(f"Righe dopo il filtro (Aprile): {len(df_mar_apr)}")

print("\nPrime 5 righe (inizio Marzo):")
print(df_mar_apr.head())
print("\nUltime 5 righe (fine Aprile):")
print(df_mar_apr.tail())

Righe totali nel DataFrame completo: 524161
Righe dopo il filtro (Aprile): 87840

Prime 5 righe (inizio Marzo):
                              price
2021-03-01 00:00:00+00:00  0.048145
2021-03-01 00:01:00+00:00  0.048209
2021-03-01 00:02:00+00:00  0.048286
2021-03-01 00:03:00+00:00  0.048241
2021-03-01 00:04:00+00:00  0.048200

Ultime 5 righe (fine Aprile):
                             price
2021-04-30 23:55:00+00:00  0.33895
2021-04-30 23:56:00+00:00  0.33920
2021-04-30 23:57:00+00:00  0.33818
2021-04-30 23:58:00+00:00  0.33829
2021-04-30 23:59:00+00:00  0.33747


In [6]:

# --- Cella 6: Salvataggio del File CSV ---

# Definiamo il nome del file di output
nome_file_output = 'prezzi_puliti_MARZO_APRILE_2.csv'

print(f"Salvataggio del file pulito in '{nome_file_output}'...")

# Salviamo il DataFrame 'df_mar_apr' (che contiene solo Aprile)
#
# 'index=True' è fondamentale qui. Dice a pandas di salvare
# anche la colonna dell'indice (che è il nostro 'open_time' in UTC).
df_mar_apr.index.name = 'open_time'
df_mar_apr.to_csv(nome_file_output, index=True)

print("Salvataggio completato!")
print(f"Troverai il file '{nome_file_output}' nella cartella del tuo notebook.")

Salvataggio del file pulito in 'prezzi_puliti_MARZO_APRILE_2.csv'...
Salvataggio completato!
Troverai il file 'prezzi_puliti_MARZO_APRILE_2.csv' nella cartella del tuo notebook.
